In [ ]:
import json
from copy import deepcopy
from sklearn.metrics import f1_score, precision_score, recall_score
from norm_metrics import compute_metrics

def get_data(path):
    with open(path, "r") as f:
        examples=[json.loads(line) for line in f]
    return examples

def form_truth_quintuples_and_triples(test_data):
    truths = []
    for example in test_data:
        events = {}
        times = {}
        event_quins = {}
        ee_trips = []
        ets = {}
        for instance in example['instances']:
            instance_id = instance["id"]
            if instance["type"] == "EVENT":
                events[instance_id] = instance
            else:
                times[instance_id] = instance

        for ee in example["ee_temprels"]:
            ee_trips.append({"event1_id":ee["e1"], "temp_relation":ee["rel"], "event2_id":ee["e2"]})

        for et in example["event_times"]:
            evid = et["event"] 
            if 'value' in times[et["time"]]:
                value = times[et["time"]]['value']
            else:
                value = None
            if evid not in ets:
                ets[evid] = [value]
            else:
                ets[evid].append(value)

        for eid, event in events.items():
            quint = {
                "id": eid,
                "event": event["text"],
                "subject": None,
                "object": None,
                "times": ets.get(eid, [])
            }
            event_quins[eid] = quint

        truths.append({'times':list(times.values()), 'quintuples':list(event_quins.values()), 'triples':list(ee_trips)})

def sample_ner_compare(truths, preds):
    preds_copy = deepcopy(preds)
    results = []
    for instance in truths:
        if instance['type'] != "EVENT" and instance['id']==0:
            continue
        text_match = False
        type_match = False
        for pred in preds_copy:
            if instance['text'] == pred[0]:
                text_match = "strict"
                type_match = instance['type'] == pred[1]
            elif instance['text'] in pred[0] or pred[0] in instance['text']:
                text_match = "relaxed"
                type_match = instance['type'] == pred[1]

            if text_match != False:
                preds_copy.remove(pred)
                results.append((text_match, type_match, instance['type']))
                break
        if text_match == False:
            results.append((text_match, type_match, instance['type']))
    return results

def sample_quintuple_compare(truths, preds):
    return

def sample_triple_compare(truths, preds):
    return

def get_ner_scores(results):
    strict_text_match = [1 if item[0]=="strict" else 0 for item in results]
    relaxed_text_match = [1 if item[0] in ["relaxed","strict"] else 0 for item in results]
    type_match = [1 if item[1]==True else 0 for item in results]

    return {"strict_text":f1_score([1]*len(strict_text_match), strict_text_match),
            "relaxed_text":f1_score([1]*len(strict_text_match), relaxed_text_match),
            "type":f1_score([1]*len(strict_text_match), type_match)}


In [2]:
test_data = get_data("D:\\GeoTKG\\cleandata\\tie\\test.json")
et_ner_preds = get_data("llama3.2-8B-ner-preds.json")

In [12]:
results = []
for pred, truth in zip(et_ner_preds, test_data):
    results.extend(sample_ner_compare(truth['instances'], pred['pred']))
get_ner_scores(results)

{'strict_text': 0.11744498121309715,
 'relaxed_text': 0.1577896837903141,
 'type': 0.14119143523425906}

In [ ]:
truths = []

for example in test_data[:1]:
    events = {}
    times = {}
    event_quins = {}
    ee_trips = []
    ets = {}
    for instance in example['instances']:
        instance_id = instance["id"]
        if instance["type"] == "EVENT":
            events[instance_id] = instance
        else:
            times[instance_id] = instance

    for ee in example["ee_temprels"]:
        ee_trips.append({"event1_id":ee["e1"], "temp_relation":ee["rel"], "event2_id":ee["e2"]})

    for et in example["event_times"]:
        evid = et["event"] 
        if 'value' in times[et["time"]]:
            value = times[et["time"]]['value']
        else:
            value = None
        if evid not in ets:
            ets[evid] = [value]
        else:
            ets[evid].append(value)

    for eid, event in events.items():
        quint = {
            "id": eid,
            "event": event["text"],
            "subject": None,
            "object": None,
            "times": ets.get(eid, [])
        }
        event_quins[eid] = quint

    truths.append({'times':list(times.values()), 'quintuples':list(event_quins.values()), 'triples':list(ee_trips)})


In [ ]:
time_scores =[]
from copy import deepcopy
def time_match(all_ground, all_pred):
    ap = deepcopy(all_pred)
    prediction_scores = []
    for t in all_ground:
        if t['id'] == 0:
            continue
        value_score = "NONE"
        match_score = "NONE"
        type_score = "NONE"
        for p in ap:
            if t['text'] in [ptext[1] for ptext in ap]:
                ap.remove(p)
                match_score = "MATCH"

                if t['value'] == p[2]:
                    value_score = "CORRECT"
                elif t['value'] is not None and p[2] is None:
                    value_score = "HALLUCINATED"
                
                if p[3] in ['DATE', 'DURATION', 'SET', 'TIME', 'GEO_TIME']:
                    type_score = p[3]
                else:
                    type_score = "HALLUCINATED"
                prediction_scores.append((match_score, value_score, type_score, t['type']))
                break
        prediction_scores.append((match_score, value_score, type_score, "NONE"))
    return prediction_scores

for truth, pred in zip(truths, out.values()):
    time_scores.extend(time_match(truth['times'], pred['pred']['times']))

time_scores